In [2]:
import numpy as np
from Track import gps_to_segments

In [3]:
x, y, segments = gps_to_segments("../track/Melbourne.csv",
    target_seg_len=50,
    kappa_straight=0.007,
    ay_max=30.0,
    hs_speed_mps=50.0,
    curv_percentile=90,
    )

In [11]:
NUM_OF_LAPS = 1
DRAG_COEFFICIENT = 0.9
DOWNFORCE_COEFFICIENT = 2.37
MASS = 800 + NUM_OF_LAPS * 1.85  # including driver, in kg

In [ ]:
# def tire_grip_coefficient(
#     temperature: float,
#     wear: float,
#     optimal_temp: float = 90.0,
#     temp_sensitivity: float = 0.01,
# ) -> float:
#     """Calculate tire grip coefficient.
    
#     Simplified model: grip peaks at optimal temperature,
#     decreases with wear and temperature deviation.
    
#     Args:
#         temperature: Tire temperature in °C
#         wear: Tire wear [0, 1] where 1 is new
#         optimal_temp: Optimal operating temperature
#         temp_sensitivity: Sensitivity to temperature deviation
        
#     Returns:
#         Grip coefficient [0, 1]
#     """
#     # Temperature factor: Gaussian around optimal
#     temp_factor = np.exp(-temp_sensitivity * (temperature - optimal_temp) ** 2)
    
#     # Wear factor: linear degradation
#     wear_factor = 0.7 + 0.3 * wear  # 70% grip at zero wear
    
#     return float(np.clip(temp_factor * wear_factor, 0.0, 1.0))



def aero_efficiency(
    velocity: float,
    frontal_area: float = 1.5,
    air_density: float = 1.225,
    mode: int = 0
) -> float: 
    
    '''
    k = 1/2 * rho * Cd * A 
    k_2026 = 2.185 (publicly-determined value in general agreement with F1 data)
        
    # Another Method
    if aero_mode == 0: # X-mode
        k = 2.185 * 0.4 # 40% reduction in downforce in straight line mode
        return k * velocity ** 2
    '''
    k = 0.5 * air_density * frontal_area

    if mode == 1: # X-mode
        k = k * DRAG_COEFFICIENT
        return k * velocity ** 2
    
    return k * DOWNFORCE_COEFFICIENT * velocity ** 2



# def tire_wear_rate(
#     slip_ratio: float,
#     slip_angle: float,
#     load: float,
#     base_rate: float = 0.0001,
# ) -> float:
#     """Calculate tire wear rate per timestep.
    
#     Wear increases with slip and load.
    
#     Args:
#         slip_ratio: Longitudinal slip ratio
#         slip_angle: Lateral slip angle in radians
#         load: Vertical load on tire in Newtons
#         base_rate: Base wear rate per timestep
        
#     Returns:
#         Wear rate [0, 1] per timestep
#     """
#     slip_factor = 1.0 + abs(slip_ratio) * 10.0 + abs(slip_angle) * 5.0
#     load_factor = load / 5000.0  # Normalize by typical F1 tire load
    
#     return base_rate * slip_factor * load_factor


def update_tire_wear():
    pass

def cornering_speed():
    pass

def battery_soc():
    pass

def battery_deploy():
    pass

def battery_recover():
    pass

def tire_degradation():
    pass

def brake_force():
    pass

def fuel_consumption_rate(
    throttle: float,
    rpm: float,
    base_rate: float = 0.0001,
) -> float:
    """Calculate fuel consumption rate per timestep.
    
    Args:
        throttle: Throttle position [0, 1]
        rpm: Engine RPM
        base_rate: Base consumption rate
        
    Returns:
        Fuel consumed [0, 1] per timestep (fraction of tank)
    """
    throttle_factor = 0.3 + 0.7 * throttle  # Idle consumption + throttle
    rpm_factor = rpm / 15000.0  # Normalize by max RPM
    
    return base_rate * throttle_factor * rpm_factor


def Update_fuel_and_mass(
        throttle: float,
        rpm: float,
):
    """
    Update fuel mass and total mass based on consumption.
    """
    fuel_consumed = (NUM_OF_LAPS * 1.85) * fuel_consumption_rate(throttle, rpm) 
    MASS -= fuel_consumed
    return MASS


def lateral_acceleration_limit(
    velocity: float,
    grip: float,
    gravity: float = 9.81,
) -> float:
    """Calculate maximum lateral acceleration.
    
    Args:
        velocity: Vehicle velocity in m/s
        grip: Tire grip coefficient [0, 1]
        downforce: Aerodynamic downforce in Newtons
        mass: Vehicle mass in kg
        gravity: Gravitational acceleration
        
    Returns:
        Maximum lateral acceleration in m/s²
    """

    downforce = aero_efficiency(velocity, aero_mode=0)  # Get downforce at current speed
    # Total vertical force = weight + downforce
    vertical_force = MASS * gravity + downforce
    
    # Maximum lateral force = grip * vertical force
    max_lateral_force = grip * vertical_force
    
    # Maximum lateral acceleration
    return max_lateral_force / MASS

In [14]:
test = aero_efficiency(53, mode=1)
print(test)

2322.691875
